# CGR-MAT v1.3 — Schema-Correct Prognostic Cohort

This version uses only confirmed appendicitis patients with an observed `Severity` label for supervised prognostic training: **445 patients, 116 complicated, 329 uncomplicated**. The additional **18 confirmed appendicitis patients with missing Severity** are exported separately and are never assigned invented labels.

The notebook mounts Google Drive before launching the pipeline, reuses the verified 523 MB archive when available, and saves every checkpoint, table, figure and result bundle to Drive.


In [ ]:
!pip -q install catboost==1.2.8 openpyxl==3.1.5


In [ ]:
from pathlib import Path
from google.colab import drive

def mount_drive_safely():
    mount_point = Path('/content/drive')
    try:
        drive.mount(str(mount_point), force_remount=False, timeout_ms=120000)
    except Exception as first_error:
        print('First Drive mount attempt failed:', first_error)
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        drive.mount(str(mount_point), force_remount=True, timeout_ms=120000)
    root = mount_point / 'MyDrive'
    if not root.exists():
        raise RuntimeError('Google Drive is not mounted. Enable pop-ups/cookies and rerun this cell.')
    print('✓ Google Drive is ready:', root)

mount_drive_safely()


In [ ]:
import os
import torch

os.environ['CGR_MAT_RUN_MODE'] = 'full'
os.environ['CGR_MAT_USE_DRIVE'] = '1'
os.environ['CGR_MAT_FORCE_RESTART'] = '0'
os.environ['CGR_MAT_PRETRAINED'] = '1'

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then rerun all cells.')
print('Run mode:', os.environ['CGR_MAT_RUN_MODE'])
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import hashlib
import urllib.request

SOURCE_COMMIT = '1ea74130ccb0010facd1f167f752b5ad3c3bf254'
LOADER_URL = (
    'https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/'
    f'{SOURCE_COMMIT}/src/cgr_mat/cgr_mat_verified_loader_v1_3.py'
)
print('Loading pinned CGR-MAT v1.3 loader...')
print('Source commit:', SOURCE_COMMIT)
loader_bytes = urllib.request.urlopen(LOADER_URL, timeout=120).read()
print('Loader SHA256:', hashlib.sha256(loader_bytes).hexdigest())
loader = loader_bytes.decode('utf-8')
exec(compile(loader, LOADER_URL, 'exec'), globals(), globals())


## Required pre-training audit

The pipeline must display:

- `supervised_confirmed_appendicitis = 445`
- `complicated = 116`
- `uncomplicated = 329`
- `confirmed_appendicitis_missing_severity = 18`
- `all_confirmed_appendicitis_by_diagnosis = 463`

It saves `prognostic_cohort_label_audit.csv` and `confirmed_appendicitis_missing_severity.csv`. All training artifacts are written under `MyDrive/MAT-Appendix/cgr_mat_runs/cgr_mat_v1_3_full_<hash>/`.
